In [2]:
import pandas as pd
import glob
import os

def load_all_logs(log_root_dir):
    all_files = glob.glob(os.path.join(log_root_dir, "**", "*.csv"), recursive=True)
    
    log_dfs = []
    for file in all_files:
        df = pd.read_csv(file)
        log_dfs.append(df)

    logs = pd.concat(log_dfs, ignore_index=True)
    return logs

In [3]:
def label_attack_flows(flow_csv, log_root_dir, output_csv):
    print(f"Labeling {flow_csv} using logs in {log_root_dir}...")
    flows = pd.read_csv(flow_csv)
    logs = load_all_logs(log_root_dir)

    

    logs["timestamp"] = pd.to_datetime(logs["Timestamp"], utc=True, errors="coerce")
    logs = logs.dropna(subset=["timestamp"])

    logs["epoch"] = logs["timestamp"].apply(lambda x: x.timestamp())
    logs["time_window"] = (logs["epoch"] * 10).astype(int)   # 100ms windows

    start, end = flows["time_window"].min(), flows["time_window"].max()
    # logs_in_range = [a for a in logs["epoch"] if ((a*10 >= start) and (a*10 <= end+1))]
    
    # # Count attacks per time_window per TargetIP
    # logs_grouped = (
    #     logs.groupby(["time_window", "TargetIP"])
    #         .agg(attack_count=("Attack", "count"))
    #         .reset_index()
    # )
    
    # print(len(logs))
    # print(logs_grouped["attack_count"].sum())

    logs_in_range_df = logs[(logs["time_window"] >= start) & (logs["time_window"] <= end)]

    logs_grouped = (
        logs_in_range_df.groupby(["time_window", "TargetIP"])
        .agg(attack_count=("Attack", "count"))
        .reset_index()
    )

    print("Logs inside flow time range:", len(logs_in_range_df))
    print("Total attacks in grouped:", logs_grouped["attack_count"].sum())

    flows["time_window"] = flows["time_window"].astype(int)

    
    merged = flows.merge(
        logs_grouped,
        left_on=["time_window", "ip.dst"],
        right_on=["time_window", "TargetIP"],
        how="left"
    )

    merged["attack_count"] = merged["attack_count"].fillna(0)

    # Binary label
    merged["is_attack"] = (merged["attack_count"] > 0).astype(int)

    # Clean up
    merged.drop(columns=["TargetIP"], inplace=True)

    print("Attack windows:", merged["is_attack"].sum())

    merged.to_csv(output_csv, index=False)
    print(f"Saved → {output_csv}\n")


In [4]:

# # Label external attack
# label_attack_flows(
#     "../train/external_flows.csv",
#     "../data/attack/external/network-wide/",   
#     "external_labeled.csv"
# )[a for a in logs["epoch"] if ((a*10 >= start) and (a*10 <= end))]

In [5]:

label_attack_flows(
    "../train/cscada_flows.csv",
    "../data/attack/compromised-scada/attack logs",    
    "../train/cscada_labeled.csv"
)

Labeling ../train/cscada_flows.csv using logs in ../data/attack/compromised-scada/attack logs...
Logs inside flow time range: 636597
Total attacks in grouped: 636597
Attack windows: 242
Saved → ../train/cscada_labeled.csv

